<a href="https://colab.research.google.com/github/Naman1232/ML-PROJECTS/blob/main/GAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from keras.datasets import mnist
from keras.layers import Input,Dense,Reshape,Flatten
from keras.layers import BatchNormalization
from keras.layers import LeakyReLU
from keras.models import Sequential,Model
from keras.optimizers import Adam
import matplotlib.pyplot as plt
import numpy as np

* **Input:** Defines the shape of the input data entering the model.
* **Dense:** A fully connected layer where every input neuron connects to every output neuron.
* **Reshape:** Changes the shape of tensors without altering the data itself.
* **Flatten:** Converts multi-dimensional data (e.g., images) into a 1D vector, required because dense layers accept only 1D inputs.
* **BatchNormalization:** Normalizes layer outputs to have zero mean and unit variance, stabilizing and speeding up training.
basically layers are connected in sequetial manner so o/p of 1 layer acts as i/p to other ,now if the o/p keeps changing a lot so the layer has to adjust everytime . That is why we normalise the o/p using batchnormalisation so as to speed up the training process.

* **LeakyReLU:** An activation function that allows a small slope for negative inputs, preventing dead neurons and improving learning. Not like normal relu which doesnt allow negative inputs at all.




In [2]:
img_rows=28
img_cols=28
channels=1
img_shape=(img_rows,img_cols,channels)

In [3]:
def build_generator():
  noise_shape=(100,)
  model=Sequential()
  model.add(Dense(256,input_shape=noise_shape))
  model.add(LeakyReLU(alpha=0.2))
  model.add(BatchNormalization(momentum=0.8))
  model.add(Dense(512))
  model.add(LeakyReLU(alpha=0.2))
  model.add(BatchNormalization(momentum=0.8))
  model.add(Dense(1024))
  model.add(LeakyReLU(alpha=0.2))
  model.add(BatchNormalization(momentum=0.8))
  model.add(Dense(np.prod(img_shape),activation='tanh'))
  model.add(Reshape(img_shape))
  model.summary()
  noise=Input(shape=noise_shape)
  img=model(noise)
  return Model(noise,img)


* Takes a **random noise vector** of size 100 as input.
* Passes noise through **Dense layers** with increasing neurons: 256 → 512 → 1024.
* Each Dense layer is followed by **LeakyReLU activation** and **BatchNormalization** for better training.
* Final Dense layer outputs a vector matching the total image size with **tanh activation** (values between -1 and 1).
* Reshapes this flat vector into the desired **image shape** (e.g., 28x28x1).
* Creates and returns a **Keras model** that converts noise input into a generated image.


In [4]:
def build_discriminator():
  model=Sequential()
  model.add(Flatten(input_shape=img_shape))
  model.add(Dense(512))
  model.add(LeakyReLU(alpha=0.2))
  model.add(Dense(256))
  model.add(LeakyReLU(alpha=0.2))
  model.add(Dense(1,activation='sigmoid'))
  model.summary()

  img=Input(shape=img_shape)
  validity=model(img)
  return Model(img,validity)


* **Purpose:** Creates a neural network called a discriminator, which decides if an input image is real or fake.
* **Input:** Takes an image with a shape defined by `img_shape`.
* **Steps inside the model:**

  * Flattens the image into a 1D vector.
  * Adds a dense (fully connected) layer with 512 neurons, followed by a LeakyReLU activation (helps learning by allowing small gradients when input is negative).
  * Adds another dense layer with 256 neurons and LeakyReLU activation.
  * Adds a final dense layer with 1 neuron and a sigmoid activation (outputs a value between 0 and 1, representing the probability that the image is real).
* **Output:** Returns a Keras Model that takes an image input and outputs a validity score (real or fake).
* **Summary:** Prints the model architecture.



In [5]:
def train(epochs,batch_size=128,save_interval=50):
  (X_train,_),(_,_)=mnist.load_data()
  X_train=(X_train.astype(np.float32)-127.5)/127.5
  X_train=np.expand_dims(X_train,axis=3)
  half_batch=int(batch_size/2)

  for epoch in range(epochs):

    idx=np.random.randint(0,X_train.shape[0],half_batch)
    imgs=X_train[idx]
    noise=np.random.normal(0,1,(half_batch,100))
    gen_imgs=generator.predict(noise)

    d_loss_real=discriminator.train_on_batch(imgs,np.ones((half_batch,1)))
    d_loss_fake=discriminator.train_on_batch(gen_imgs,np.zeros((half_batch,1)))

    d_loss=0.5*np.add(d_loss_real,d_loss_fake)

    noise=np.random.normal(0,1,(batch_size,100))
    valid_y=np.array([1]*batch_size)
    g_loss=combined.train_on_batch(noise,valid_y)

    print("%d [D loss:%f,acc.:%.2f%%][G loss:%f]"%(epoch,d_loss[0],100*d_loss[1],g_loss))
    if epoch % save_interval == 0:
            save_imgs(epoch)



### **Step 1: Load and Prepare the Data**

* Load the MNIST dataset.
* Normalize the pixel values from \[0, 255] to \[-1, 1] to match the output of the generator (which uses a `tanh` activation).
* Expand the image dimensions to include the channel (since these are grayscale images with shape 28×28×1).

---

### **Step 2: Define Half Batch Size**

* Divide the full batch size by 2, because the discriminator will be trained on half real and half fake images in each iteration.

---

### **Step 3: Start Training Loop**

* For each epoch in training:

---

#### 🔹 **3.1: Sample Real Images**

* Randomly select a batch of real images from the dataset to train the discriminator.

---

#### 🔹 **3.2: Generate Fake Images**

* Create random noise vectors.
* Use the generator to convert the noise into fake images.

---

#### 🔹 **3.3: Train the Discriminator**

* Train the discriminator on:

  * Real images labeled as "real" (1)
  * Generated fake images labeled as "fake" (0)
* Average the loss from both the real and fake samples to get the final discriminator loss for the batch.

---

#### 🔹 **3.4: Train the Generator**

* Create another batch of noise vectors.
* Label the fake images as "real" (1) intentionally.
* Train the **combined model** (generator + frozen discriminator) to adjust the generator’s weights, so that it can fool the discriminator into thinking its fake images are real.

---

#### 🔹 **3.5: Print and Save Results**

* Print the current losses and accuracy of the discriminator, and the loss of the generator.
* Every few epochs (based on a save interval), save sample generated images to track progress.



In [6]:
import os

def save_imgs(epoch):
    print(f"[INFO] Saving images for epoch {epoch}...")

    r, c = 5, 5
    noise = np.random.normal(0, 1, (r * c, 100))
    gen_imgs = generator.predict(noise)

    gen_imgs = 0.5 * gen_imgs + 0.5  # Rescale to [0, 1]

    fig, axs = plt.subplots(r, c)
    cnt = 0
    for i in range(r):
        for j in range(c):
            axs[i, j].imshow(gen_imgs[cnt, :, :, 0], cmap='gray')
            axs[i, j].axis('off')
            cnt += 1

    save_path = "/content/images"
    os.makedirs(save_path, exist_ok=True)

    save_file = os.path.join(save_path, f"mnist_{epoch}.png")
    fig.savefig(save_file)
    plt.close()

    print(f"[INFO] Image saved to: {save_file}")


1. Grid Size	Sets up a 5×5 grid for displaying images
1. Generate Noise	Creates 25 random noise vectors (each size 100)
1. Generate Images	Generator turns those noise vectors into fake images
1. Rescale Images	Converts pixel values from [-1, 1] to [0, 1] for viewing
2. Plot Images	Displays all 25 images in a grid using matplotlib
2. Save Plot	Saves the plotted image grid as a PNG file with the current epoch number




In [7]:
optimizer = Adam(0.0002, 0.5)

* **Adam** is an optimizer that helps the model learn by adjusting weights efficiently using gradients.
* **0.0002** is the learning rate, which controls how big each step is when updating weights.
* The 0.5 (beta_1) means the optimizer balances between old and new gradient information when updating weights — it neither forgets the past too quickly nor relies only on the latest changes.

This helps smooth out updates and makes training more stable


In [8]:
discriminator = build_discriminator()
discriminator.compile(loss='binary_crossentropy',
    optimizer=optimizer,
    metrics=['accuracy'])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       401,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 533,505 (2.04 MB)

 Trainable params: 533,505 (2.04 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
generator = build_generator()
generator.compile(loss='binary_crossentropy', optimizer=optimizer)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 256)            │        25,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_3 (LeakyReLU)       │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1024)           │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_4 (LeakyReLU)       │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 784)            │       803,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 28, 28, 1)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,493,520 (5.70 MB)

 Trainable params: 1,489,936 (5.68 MB)

 Non-trainable params: 3,584 (14.00 KB)

In [10]:
z = Input(shape=(100,))
img = generator(z)

In [11]:
discriminator.trainable = False
valid = discriminator(img)

while training the generator we keep the discriminator constant coz generator works on a fixed output given by discriminator so that should not change while the generator is training.

In [12]:
combined = Model(z, valid)
combined.compile(loss='binary_crossentropy', optimizer=optimizer)


so generator basically generates fake images which are then passed through the discriminator which basically tells whether it is real or fake.


In [13]:
train(epochs=3000, batch_size=32, save_interval=300)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 840ms/step


/usr/local/lib/python3.11/dist-packages/keras/src/backend/tensorflow/trainer.py:82: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


Streaming output truncated to the last 5000 lines.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
512 [D loss:2.635808,acc.:9.50%][G loss:0.048911]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
513 [D loss:2.637353,acc.:9.48%][G loss:0.048823]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
514 [D loss:2.638801,acc.:9.49%][G loss:0.048734]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
515 [D loss:2.640072,acc.:9.50%][G loss:0.048647]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
516 [D loss:2.641226,acc.:9.52%][G loss:0.048561]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
517 [D loss:2.642300,acc.:9.51%][G loss:0.048475]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
518 [D loss:2.643506,acc.:9.52%][G loss:0.048389]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
519 [D loss:2.644840,acc.:9.51%][G loss:0.048304]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
520 [D loss:2.646379,acc.:9.50%][G loss:0.048219]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
521 [D loss:2.647605,acc.:9.51%][G loss:0.048133]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
522 [D loss:2.648685,acc.:9.51%

In [14]:

!zip -r /content/images.zip /content/images


from google.colab import files
files.download('/content/images.zip')


  adding: content/images/ (stored 0%)
  adding: content/images/mnist_1800.png (deflated 11%)
  adding: content/images/mnist_2400.png (deflated 11%)
  adding: content/images/mnist_600.png (deflated 11%)
  adding: content/images/mnist_900.png (deflated 11%)
  adding: content/images/mnist_1200.png (deflated 11%)
  adding: content/images/mnist_2700.png (deflated 11%)
  adding: content/images/mnist_300.png (deflated 11%)
  adding: content/images/mnist_1500.png (deflated 11%)
  adding: content/images/mnist_2100.png (deflated 11%)
  adding: content/images/mnist_0.png (deflated 7%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:

generator.save('generator_model.h5')